# CodeT5 Fine-tuning for Python Line Completion

The LSTM in `03_lstm_model` was trained from scratch. Here we fine-tune **CodeT5-small**
(`Salesforce/codet5-small`, ~60M parameters), an encoder-decoder transformer that has already been
pretrained on a large amount of source code.

**Task:** given the code before the cursor, generate the **rest of the current line**.

**Data:** the `(prefix, target)` pairs built in `04_codet5_data` and saved to Google Drive:

- `prefix` — up to 1000 characters of code before the cursor (comments/docstrings removed, original spacing kept)
- `target` — the rest of that line

Unlike the LSTM, CodeT5 works on **raw code text** with its own sub-word tokenizer, so there is no
fixed vocabulary and no `<UNK>` token.

In [ ]:
!pip install -q transformers

In [ ]:
import os
import json
import difflib
import random

import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, T5ForConditionalGeneration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Config

In [ ]:
DATA_DIR = "/content/drive/MyDrive/code_completion"     # same folder as 04_codet5_data
BEST_MODEL_DIR = os.path.join(DATA_DIR, "codet5_best")  # saved on Drive so it survives runtime resets

MODEL_NAME = "Salesforce/codet5-small"

MAX_SOURCE_LEN = 256   # sub-word tokens of context (~1000 characters)
MAX_TARGET_LEN = 64    # targets are at most 150 characters

BATCH_SIZE = 32
LEARNING_RATE = 1e-4
NUM_EPOCHS = 3
PATIENCE = 2

random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Load the pairs

In [ ]:
def load_jsonl(name):
    with open(os.path.join(DATA_DIR, name)) as f:
        return [json.loads(line) for line in f]

train_pairs = load_jsonl("train.jsonl")
valid_pairs = load_jsonl("valid.jsonl")
test_pairs = load_jsonl("test.jsonl")

print(len(train_pairs), len(valid_pairs), len(test_pairs))

In [ ]:
example = train_pairs[0]

print(example["prefix"][-200:] + "⟦CURSOR⟧" + example["target"])

## Tokenizer

The cursor is at the **end** of the prefix, so the most useful context is the code right before it.
`truncation_side = "left"` makes long prefixes lose their *beginning* instead of their end.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.truncation_side = "left"

print(tokenizer.tokenize(example["target"]))

## Dataset + DataLoader

The dataset just returns the raw strings; `collate_fn` tokenizes a whole batch at once and pads it.
Padding positions in the labels are set to `-100` so the loss ignores them.

In [ ]:
class CodeT5Dataset(Dataset):

    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return self.pairs[idx]["prefix"], self.pairs[idx]["target"]


def collate_fn(batch):
    prefixes, targets = zip(*batch)

    encoder = tokenizer(
        list(prefixes),
        max_length=MAX_SOURCE_LEN,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    labels = tokenizer(
        list(targets),
        max_length=MAX_TARGET_LEN,
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).input_ids

    labels[labels == tokenizer.pad_token_id] = -100

    return encoder.input_ids, encoder.attention_mask, labels

In [ ]:
train_loader = DataLoader(
    CodeT5Dataset(train_pairs),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

valid_loader = DataLoader(
    CodeT5Dataset(valid_pairs),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    CodeT5Dataset(test_pairs),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

In [ ]:
input_ids, attention_mask, labels = next(iter(train_loader))

print(input_ids.shape)
print(attention_mask.shape)
print(labels.shape)

## Model

When we pass `labels`, the model shifts them right internally to build the decoder input
(teacher forcing) and returns the cross-entropy loss.

In [ ]:
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)

print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

## Training

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()

    epoch_loss = 0.0

    for input_ids, attention_mask, labels in tqdm(loader, leave=False):

        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(loader)

In [ ]:
def validate(model, loader, device):
    model.eval()

    total_loss = 0.0

    with torch.no_grad():
        for input_ids, attention_mask, labels in loader:

            outputs = model(
                input_ids=input_ids.to(device),
                attention_mask=attention_mask.to(device),
                labels=labels.to(device)
            )

            total_loss += outputs.loss.item()

    return total_loss / len(loader)

In [ ]:
best_val_loss = float("inf")
epochs_without_improvement = 0

train_losses = []
val_losses = []

for epoch in range(NUM_EPOCHS):

    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_loss = validate(model, valid_loader, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(
        f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] "
        f"| Train Loss: {train_loss:.4f} "
        f"| Val Loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0

        model.save_pretrained(BEST_MODEL_DIR)
        tokenizer.save_pretrained(BEST_MODEL_DIR)
        print("   ✓ Best model saved")

    else:
        epochs_without_improvement += 1
        print(f"   No improvement ({epochs_without_improvement}/{PATIENCE})")

        if epochs_without_improvement >= PATIENCE:
            print("Early stopping triggered.")
            break

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CodeT5 Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

## Evaluation

The model generates each line on its own (greedy decoding). We use the two standard line-completion metrics
from the CodeXGLUE benchmark:

- **Exact match** — the generated line is exactly the true line (ignoring surrounding whitespace)
- **Edit similarity** — how close the two strings are character by character (0–100), so an
  almost-correct line still gets partial credit

In [ ]:
model = T5ForConditionalGeneration.from_pretrained(BEST_MODEL_DIR).to(device)

test_loss = validate(model, test_loader, device)
print(f"Test Loss: {test_loss:.4f}")

In [ ]:
def generate_lines(model, prefixes, batch_size=64):
    model.eval()
    predictions = []

    for start in tqdm(range(0, len(prefixes), batch_size), leave=False):
        encoder = tokenizer(
            prefixes[start:start+batch_size],
            max_length=MAX_SOURCE_LEN,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            generated = model.generate(**encoder, max_length=MAX_TARGET_LEN)

        predictions += tokenizer.batch_decode(
            generated,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

    return predictions


def line_metrics(predictions, targets):
    exact_matches = 0
    edit_similarity = 0.0

    for pred, target in zip(predictions, targets):
        pred, target = pred.strip(), target.strip()

        exact_matches += pred == target
        edit_similarity += difflib.SequenceMatcher(None, pred, target).ratio()

    return exact_matches / len(targets), 100 * edit_similarity / len(targets)

In [ ]:
predictions = generate_lines(model, [p["prefix"] for p in test_pairs])
exact_match, edit_similarity = line_metrics(predictions, [p["target"] for p in test_pairs])

print("\n===== FINAL CodeT5 TEST RESULTS =====")
print(f"Test Loss        : {test_loss:.4f}")
print(f"Exact Match      : {exact_match * 100:.2f}%")
print(f"Edit Similarity  : {edit_similarity:.2f}")

In [ ]:
# a few test examples: true line vs prediction
for i in random.sample(range(len(test_pairs)), 5):
    print("-" * 60)
    print(test_pairs[i]["prefix"][-120:] + "⟦CURSOR⟧")
    print("true :", repr(test_pairs[i]["target"]))
    print("pred :", repr(predictions[i]))

## Autocomplete demo

No lexer needed at inference time: the raw code before the cursor is the model input.

In [ ]:
def complete(code):
    encoder = tokenizer(
        code,
        max_length=MAX_SOURCE_LEN,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        generated = model.generate(**encoder, max_length=MAX_TARGET_LEN)

    return tokenizer.decode(
        generated[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

In [ ]:
prompts = [
    "def add(a, b):\n    return",
    "import numpy as",
    "for i in range(",
    "with open(path, 'r') as f:\n    data =",
    "if __name__ ==",
]

for prompt in prompts:
    print(prompt + "⟦" + complete(prompt) + "⟧")
    print()